# From features to win-probability model

Builds the 2nd-innings ball-by-ball dataset (`src/features.py`), explains the
time-aware validation design, and reproduces the reported model results with
the saved artifacts (`src/train.py`).

## Setup and data

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from src.features import build_chase_features, LABEL
from src.train import season_split, TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS
from src import evaluate as ev

plt.rcParams["figure.dpi"] = 110

csv = ROOT / "data" / "processed" / "chase_features.csv"
if not csv.exists():
    from src.data import clean_deliveries, clean_matches, load_deliveries, load_matches
    feats = build_chase_features(clean_matches(load_matches()), clean_deliveries(load_deliveries()))
else:
    feats = pd.read_csv(csv)
parts = season_split(feats)
print(f"feature rows: {len(feats):,} across {feats['match_id'].nunique()} matches")
for name, part in parts.items():
    print(f"{name:>10}: seasons {part['season_year'].min()}-{part['season_year'].max()}, "
          f"{part['match_id'].nunique()} matches, {len(part):,} balls")

## No-future-leakage and split design checks

- Every feature is computed from the state **at that ball** (cumulative runs,
  legal balls, wickets, rolling win rates from *earlier* matches only). The
  target is DLS-adjusted and known before the chase starts.
- The split is **by season** (train 2008-2021, validation 2022, test
  2023-2024); `season_split` raises if a match id ever appears twice, and the
  call above ran clean.
- Unit tests in `tests/test_features.py` prove: a wide never advances the ball
  count, retired hurt never counts as a wicket, mutating a later ball does
  not change earlier features, and flipping the winner changes only the label.

## Model comparison (all tuned on the 2022 validation season only)

In [ ]:
meta = json.loads((ROOT / "models" / "metadata.json").read_text())
res = meta["results"]
rows = [{
    "model": "naive: RRR <= 9.0 -> win",
    "val_log_loss": res["naive_rrr_threshold"]["val_log_loss"],
    "test_log_loss": res["naive_rrr_threshold"]["log_loss"],
    "test_brier": res["naive_rrr_threshold"]["brier"],
    "test_roc_auc": res["naive_rrr_threshold"]["roc_auc"],
}]
for fam in ("logreg", "random_forest", "xgboost"):
    rows.append({
        "model": f"{fam} (val-tuned)",
        "val_log_loss": res[fam]["val_log_loss"],
        "test_log_loss": np.nan, "test_brier": np.nan, "test_roc_auc": np.nan,
    })
fin = res["final_model"]
rows.append({
    "model": f"FINAL: {fin['family']} {fin['params']}",
    "val_log_loss": fin["val_log_loss"],
    "test_log_loss": fin["test_uncalibrated"]["log_loss"],
    "test_brier": fin["test_uncalibrated"]["brier"],
    "test_roc_auc": fin["test_uncalibrated"]["roc_auc"],
})
comparison = pd.DataFrame(rows)
print(comparison.round(4).to_string(index=False))
print("\nmomentum ablation:", res["momentum_ablation"]["decision"],
      f"(delta {res['momentum_ablation']['delta']:+.5f})")
print("calibration gate:", fin["calibration_gate"], "->", fin["calibrator"])

## Test-set detail: by stage of the chase and calibration

The deployed model is the un-calibrated logistic regression: calibration was
attempted (isotonic and Platt, fitted on out-of-fold training predictions) and
**rejected** by a paired statistical gate on the validation season - neither
beat the raw model beyond noise, and an isotonic fit on a single season made
test log loss *worse* (0.482 -> 0.763), a good example of calibration
overfitting a small validation set.

In [ ]:
from src.predict import load_model

art = load_model()
pipe = art["pipeline"]
fin_cols = art["feature_columns"]
X_test, y_test = parts["test"][fin_cols], parts["test"][LABEL]
p_test = pipe.predict_proba(X_test)[:, 1]

stages = ev.stage_table(parts["test"], y_test, p_test)
print(stages.round(4).to_string(index=False))
print("\ncalibration table (test):")
print(ev.calibration_table(y_test, p_test).round(3).to_string(index=False))

ev.plot_calibration({"deployed model": (y_test, p_test)},
                    ROOT / "reports" / "figures" / "calibration_curve.png")

Reading the calibration table: the model is well calibrated from ~0.45
upward, but at the low end it is **underconfident about the era**: chases with
a predicted ~3% chance actually won ~10% of the time in 2023-24. This is a
distribution shift, not a fitting bug - the impact-player rule (2023+) made
high chases easier than anything in the 2008-2021 training data. It is
documented as a limitation.

## What drives the prediction

In [ ]:
importance = pd.Series(meta["feature_importance"]).sort_values()
fig, ax = plt.subplots(figsize=(6.8, 4.4))
importance.plot.barh(ax=ax, color="#1f77b4")
ax.set_xlabel("importance share (normalized |coef|)")
ax.set_title("Feature importance of the deployed model")
fig.tight_layout()
fig.savefig(ROOT / "reports" / "figures" / "feature_importance.png", bbox_inches="tight")
plt.show()

**Plain English.** The model leans first on *where the game is played*
(venue one-hots carry a large share of the weight - grounds like Chepauk
defend far better than the aggregate), then on *who is batting and bowling*.
Among the in-play numbers, the required run rate, runs left and wickets in
hand dominate - which is exactly what a cricket fan would say watching the
game. The rolling two-year form of both teams adds a small but real signal.

## Replay: win probability through one real chase


In [ ]:
mid = (parts["test"].groupby("match_id")["chase_won"].agg(["mean", "size"])
       .query("size > 100").index)
example_id = int(parts["test"][parts["test"]["match_id"].isin(mid)]["match_id"].iloc[0])
demo = parts["test"][parts["test"]["match_id"] == example_id]
p_demo = pipe.predict_proba(demo[fin_cols])[:, 1]
wickets = demo.loc[demo["wickets_out"].diff() > 0, "balls_bowled"].tolist()

meta_m = feats.drop_duplicates("match_id").set_index("match_id")
title = f"match {example_id}: {demo['batting_team'].iloc[0]} chasing {int(demo['target'].iloc[0])}"
ev.plot_win_curve(demo["balls_bowled"].tolist(), p_demo.tolist(), wickets,
                  ROOT / "reports" / "figures" / "win_probability_replay.png", title=title)
print(title, "| final outcome:", "chase won" if demo[LABEL].iloc[-1] == 1 else "chase lost")
print("The Streamlit app renders this interactively for any 2023-24 match.")

## Limitations

- No player-level strength (form, match-ups), no weather, dew or pitch report.
- Rolling team win rates are coarse (no opponent or venue adjustment).
- The 2023-24 "impact player" era shifted scoring; the model trained through
  2021 is slightly underconfident about very hard chases in the test seasons.
- Ties (super over) and no-result matches are excluded by design.
- Not for betting or gambling - a learning/portfolio project.